In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

In [2]:
# === 1. Device Configuration ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
# === 2. Data Preparation (CIFAR-10) ===
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # ResNet input size
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [4]:
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
val_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False)


100%|██████████| 170M/170M [00:04<00:00, 40.2MB/s]


In [5]:
num_classes = 10  # CIFAR-10 has 10 categories

In [6]:
# === 3. Load Pretrained ResNet18 ===
model = models.resnet18(pretrained=True)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 112MB/s]


In [7]:
# Freeze all layers
for param in model.parameters():
    param.requires_grad = False

# Replace the final layer (unfrozen)
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

# === 4. Define Loss and Optimizer ===
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [8]:
# === 5. Training Function ===
def train_model(model, train_loader, criterion, optimizer, epochs=5):
    model.train()
    for epoch in range(epochs):
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        accuracy = 100 * correct / total
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss:.4f}, Accuracy: {accuracy:.2f}%")

In [9]:

# === 6. Evaluation Function ===
def evaluate_model(model, val_loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f'Validation Accuracy: {accuracy:.2f}%')

In [10]:
# === 7. Run Training and Evaluation ===
train_model(model, train_loader, criterion, optimizer, epochs=10)
evaluate_model(model, val_loader)

Epoch [1/10], Loss: 645.6727, Accuracy: 73.24%
Epoch [2/10], Loss: 484.7932, Accuracy: 78.64%
Epoch [3/10], Loss: 463.5932, Accuracy: 79.44%
Epoch [4/10], Loss: 449.7292, Accuracy: 80.02%
Epoch [5/10], Loss: 446.0666, Accuracy: 80.21%
Epoch [6/10], Loss: 441.1175, Accuracy: 80.52%
Epoch [7/10], Loss: 438.2872, Accuracy: 80.59%
Epoch [8/10], Loss: 430.8813, Accuracy: 80.91%
Epoch [9/10], Loss: 428.9901, Accuracy: 80.98%
Epoch [10/10], Loss: 426.1904, Accuracy: 81.09%
Validation Accuracy: 80.70%


In [11]:
# === 8. Save Model ===
torch.save(model.state_dict(), "fine_tuned_resnet18_cifar10.pth")
print("Model saved as 'fine_tuned_resnet18_cifar10.pth'")

Model saved as 'fine_tuned_resnet18_cifar10.pth'
